# 03. Sequence & Feature Encoding Pipeline

Consolidated sequence encoding pipeline for PTM site prediction.
Supports four encoding representations:
1. **One-Hot Encoding**: Position-specific encoding of sequence context and categorical residues.
2. **BLOSUM62**: Evolutionary substitution scoring matrix (31 positions × 20 amino acids = 620 features).
3. **AAPC (Amino Acid Pair Composition)**: Dipeptide frequency distribution (20 × 20 = 400 features).
4. **Hybrid Encoding**: Combines physicochemical features with sequence matrix representations.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
from pathlib import Path

print("✓ Libraries imported successfully.")

In [ ]:
# ============================================================================
# CONFIGURATION PARAMETERS
# ============================================================================

# Choose encoding method to run: "onehot", "blosum", "aapc", "hybrid", or "all"
ENCODING_METHOD = "all"

# Input dataset with base engineered features (from 02_feature_engineering)
INPUT_FILE = "../data_engineered/train_with_features.csv"
OUTPUT_BASE_DIR = "../data_engineered"

CHUNK_SIZE = 10000
MAX_LENGTH = 31

# Categorical context columns to encode for one-hot
CATEGORICAL_COLS = ['middle_aa', 'cys_ctx_-2', 'cys_ctx_-1', 'cys_ctx_1', 'cys_ctx_2']

print(f"Configuration:")
print(f"  Encoding Method: {ENCODING_METHOD}")
print(f"  Input File:      {INPUT_FILE}")
print(f"  Output Base:     {OUTPUT_BASE_DIR}")
print(f"  Chunk Size:      {CHUNK_SIZE:,}")

In [ ]:
# BLOSUM62 substitution matrix
# Rows/Columns: A  R  N  D  C  Q  E  G  H  I  L  K  M  F  P  S  T  W  Y  V
BLOSUM62 = {
    'A': [ 4, -1, -2, -2,  0, -1, -1,  0, -2, -1, -1, -1, -1, -2, -1,  1,  0, -3, -2,  0],
    'R': [-1,  5,  0, -2, -3,  1,  0, -2,  0, -3, -2,  2, -1, -3, -2, -1, -1, -3, -2, -3],
    'N': [-2,  0,  6,  1, -3,  0,  0,  0,  1, -3, -3,  0, -2, -3, -2,  1,  0, -4, -2, -3],
    'D': [-2, -2,  1,  6, -3,  0,  2, -1, -1, -3, -4, -1, -3, -3, -1,  0, -1, -4, -3, -3],
    'C': [ 0, -3, -3, -3,  9, -3, -4, -3, -3, -1, -1, -3, -1, -2, -3, -1, -1, -2, -2, -1],
    'Q': [-1,  1,  0,  0, -3,  5,  2, -2,  0, -3, -2,  1,  0, -3, -1,  0, -1, -2, -1, -2],
    'E': [-1,  0,  0,  2, -4,  2,  5, -2,  0, -3, -3,  1, -2, -3, -1,  0, -1, -3, -2, -2],
    'G': [ 0, -2,  0, -1, -3, -2, -2,  6, -2, -4, -4, -2, -3, -3, -2,  0, -2, -2, -3, -3],
    'H': [-2,  0,  1, -1, -3,  0,  0, -2,  8, -3, -3, -1, -2, -1, -2, -1, -2, -2,  2, -3],
    'I': [-1, -3, -3, -3, -1, -3, -3, -4, -3,  4,  2, -3,  1,  0, -3, -2, -1, -3, -1,  3],
    'L': [-1, -2, -3, -4, -1, -2, -3, -4, -3,  2,  4, -2,  2,  0, -3, -2, -1, -2, -1,  1],
    'K': [-1,  2,  0, -1, -3,  1,  1, -2, -1, -3, -2,  5, -1, -3, -1,  0, -1, -3, -2, -2],
    'M': [-1, -1, -2, -3, -1,  0, -2, -3, -2,  1,  2, -1,  5,  0, -2, -1, -1, -1, -1,  1],
    'F': [-2, -3, -3, -3, -2, -3, -3, -3, -1,  0,  0, -3,  0,  6, -4, -2, -2,  1,  3, -1],
    'P': [-1, -2, -2, -1, -3, -1, -1, -2, -2, -3, -3, -1, -2, -4,  7, -1, -1, -4, -3, -2],
    'S': [ 1, -1,  1,  0, -1,  0,  0,  0, -1, -2, -2,  0, -1, -2, -1,  4,  1, -3, -2, -2],
    'T': [ 0, -1,  0, -1, -1, -1, -1, -2, -2, -1, -1, -1, -1, -2, -1,  1,  5, -2, -2,  0],
    'W': [-3, -3, -4, -4, -2, -2, -3, -2, -2, -3, -2, -3, -1,  1, -4, -3, -2, 11,  2, -3],
    'Y': [-2, -2, -2, -3, -2, -1, -2, -3,  2, -1, -1, -2, -1,  3, -3, -2, -2,  2,  7, -1],
    'V': [ 0, -3, -3, -3, -1, -2, -2, -3, -3,  3,  1, -2,  1, -1, -2, -2,  0, -3, -1,  4]
}

AA_ORDER = list("ARNDCQEGHILKMFPSTWYV")  # Order of AAs in matrix
AA_TO_IDX = {aa: i for i, aa in enumerate(AA_ORDER)}

print("✓ BLOSUM62 matrix loaded")
print(f"  Matrix size: 20 × 20")
print(f"  Amino acids: {AA_ORDER}")

def sequence_to_blosum(sequence, max_length=31):
    """Encode a single sequence using BLOSUM62 matrix (max_length * 20 features)."""
    blosum_matrix = np.zeros((max_length, 20), dtype=np.float32)
    for i, aa in enumerate(sequence[:max_length]):
        if aa in BLOSUM62:
            blosum_matrix[i, :] = BLOSUM62[aa]
    return blosum_matrix.flatten()

In [ ]:
# Standard 20 amino acids for AAPC
AMINO_ACIDS = list("ACDEFGHIKLMNPQRSTVWY")
AA_TO_IDX = {aa: i for i, aa in enumerate(AMINO_ACIDS)}

def sequence_to_aapc(sequence):
    """Encode a sequence using Amino Acid Pair Composition (400 dipeptide features)."""
    aapc_vector = np.zeros(400, dtype=np.float32)
    if len(sequence) < 2:
        return aapc_vector
    
    total_pairs = len(sequence) - 1
    for i in range(total_pairs):
        aa1 = sequence[i]
        aa2 = sequence[i + 1]
        if aa1 in AA_TO_IDX and aa2 in AA_TO_IDX:
            idx = AA_TO_IDX[aa1] * 20 + AA_TO_IDX[aa2]
            aapc_vector[idx] += 1
            
    if total_pairs > 0:
        aapc_vector /= total_pairs
    return aapc_vector

In [ ]:
def get_categorical_vocab(input_file, categorical_cols):
    """Scan input file to collect global unique values for categorical columns."""
    unique_vals = {col: set() for col in categorical_cols}
    for chunk in pd.read_csv(input_file, chunksize=10000):
        for col in categorical_cols:
            if col in chunk.columns:
                unique_vals[col].update(chunk[col].dropna().unique())
    return {col: sorted(list(vals)) for col, vals in unique_vals.items()}

def encode_onehot(chunk, unique_vocab, categorical_cols, label_cols, metadata_cols):
    """One-hot encode categorical context columns while preserving metadata."""
    meta = chunk[metadata_cols]
    feats = chunk.drop(columns=metadata_cols)
    
    encoded_parts = []
    for col in categorical_cols:
        if col in feats.columns:
            dummies = pd.get_dummies(feats[col], prefix=col)
            for val in unique_vocab[col]:
                expected = f"{col}_{val}"
                if expected not in dummies.columns:
                    dummies[expected] = 0
            encoded_parts.append(dummies)
            feats = feats.drop(columns=[col])
            
    return pd.concat([meta, feats] + encoded_parts, axis=1)

In [ ]:
def encode_blosum_chunk(chunk, max_length=31):
    """Encode sequence column of chunk into BLOSUM features."""
    feature_names = [f"blosum_pos{pos}_{aa}" for pos in range(max_length) for aa in AMINO_ACIDS]
    vectors = [sequence_to_blosum(seq, max_length) for seq in chunk['Sequence']]
    blosum_df = pd.DataFrame(vectors, columns=feature_names, index=chunk.index)
    return pd.concat([chunk, blosum_df], axis=1)

def encode_aapc_chunk(chunk):
    """Encode sequence column of chunk into AAPC features."""
    feature_names = [f"aapc_{aa1}_{aa2}" for aa1 in AMINO_ACIDS for aa2 in AMINO_ACIDS]
    vectors = [sequence_to_aapc(seq) for seq in chunk['Sequence']]
    aapc_df = pd.DataFrame(vectors, columns=feature_names, index=chunk.index)
    return pd.concat([chunk, aapc_df], axis=1)

In [ ]:
def run_encoding_pipeline(method, input_file, output_base, chunk_size=10000):
    """Executes specified encoding and saves to target directory."""
    output_dir = os.path.join(output_base, method)
    os.makedirs(output_dir, exist_ok=True)
    output_file = os.path.join(output_dir, f"train_with_features_{method}.csv")
    
    print(f"\n{'='*70}\nRUNNING ENCODING: {method.upper()}\n{'='*70}")
    print(f"Input:  {input_file}")
    print(f"Output: {output_file}")
    
    label_cols = ['S-glutathionylation', 'S-nitrosylation', 'S-palmitoylation']
    metadata_cols = ['ID', 'Sequence'] + label_cols
    
    if method == 'onehot':
        vocab = get_categorical_vocab(input_file, CATEGORICAL_COLS)
    
    first_chunk = True
    total_processed = 0
    
    for chunk_idx, chunk in enumerate(pd.read_csv(input_file, chunksize=chunk_size)):
        if method == 'onehot':
            encoded_chunk = encode_onehot(chunk, vocab, CATEGORICAL_COLS, label_cols, metadata_cols)
        elif method == 'blosum':
            encoded_chunk = encode_blosum_chunk(chunk, MAX_LENGTH)
        elif method == 'aapc':
            encoded_chunk = encode_aapc_chunk(chunk)
        elif method == 'hybrid':
            # Combine physicochemical features with BLOSUM representations
            encoded_chunk = encode_blosum_chunk(chunk, MAX_LENGTH)
        else:
            raise ValueError(f"Unknown encoding method: {method}")
            
        encoded_chunk.to_csv(output_file, mode='w' if first_chunk else 'a', header=first_chunk, index=False)
        first_chunk = False
        total_processed += len(chunk)
        print(f"  [Chunk {chunk_idx+1}] Processed {len(chunk):,} rows (Total: {total_processed:,})")
        
    print(f"✓ Saved {method} encoded data to: {output_file}")
    return output_file

In [ ]:
# Execute requested encoding(s)
methods_to_run = ["onehot", "blosum", "aapc", "hybrid"] if ENCODING_METHOD == "all" else [ENCODING_METHOD]

for m in methods_to_run:
    if os.path.exists(INPUT_FILE):
        out_path = run_encoding_pipeline(m, INPUT_FILE, OUTPUT_BASE_DIR, CHUNK_SIZE)
    else:
        print(f"⚠️ Input file not found: {INPUT_FILE}")

In [ ]:
# Verify generated files
print("\n" + "="*70 + "\nVERIFICATION\n" + "="*70)
for m in ["onehot", "blosum", "aapc", "hybrid"]:
    p = os.path.join(OUTPUT_BASE_DIR, m, f"train_with_features_{m}.csv")
    if os.path.exists(p):
        df_head = pd.read_csv(p, nrows=2)
        print(f"✓ {m:<8}: Exists ({df_head.shape[1]} columns)")
    else:
        print(f"  {m:<8}: Not generated")